# 01 — Python Core Syntax
### Functions, type hints, f-strings, control flow

This is notebook 1 of a 7-part series teaching the Python you need to read
and extend the **original SupportPilot project** (the non-LangChain
version — plain functions, `sqlite3`, `re`, and Pydantic only). Each
notebook lives in the project folder so you can `import` the real modules
directly and see these concepts in actual use, not toy examples.

**Series roadmap:**
1. **Core syntax** — functions, type hints, f-strings, control flow *(this one)*
2. Data structures — dict/list/set/tuple, comprehensions
3. OOP — classes, dataclasses, Enums
4. Modules, packages & project structure
5. Regular expressions
6. Files, JSON, SQLite & the CLI (`argparse`)
7. Reading the real project end to end

No prior Python needed beyond knowing what a variable and a `print()` are.

## 1.1 Functions

A function bundles reusable logic behind a name. Every file in the project
is built almost entirely out of small, single-purpose functions.

In [1]:
def greet(name):
    return f"Hello, {name}!"

print(greet("Ananya"))


Hello, Ananya!


## 1.2 Type hints

Type hints (`name: str`, `-> str`) don't change how the code *runs* — Python
ignores them at runtime. They're documentation that tools (and humans) can
check. `escalation_rules.py`'s functions are hinted end to end:

```python
def check_hard_escalation(issue_type: str, sentiment: str, urgency: str,
                           order_amount: float = 0.0) -> tuple[bool, str]:
    ...
```

Reading this signature *without running it* already tells you: it takes
three strings and an optional float, and returns a tuple of (bool, str).
That's the whole point — the hint is a promise about shape.

In [2]:
def check_hard_escalation(issue_type: str, sentiment: str, urgency: str,
                           order_amount: float = 0.0) -> tuple[bool, str]:
    if issue_type == "fraud_suspected":
        return True, "always escalate fraud"
    return False, ""

print(check_hard_escalation("fraud_suspected", "neutral", "high"))
print(check_hard_escalation("order_status", "neutral", "low"))


(True, 'always escalate fraud')
(False, '')


### Modern union syntax: `str | None`

You'll see `str | None` everywhere instead of the older `Optional[str]` —
they mean the same thing. This project uses the newer syntax throughout
(enabled via `from __future__ import annotations` at the top of each file,
which is what makes this syntax work on slightly older Python versions
too).

In [3]:
def find_customer_name(customer_id: str | None) -> str | None:
    if customer_id is None:
        return None
    return f"Customer-{customer_id}"

print(find_customer_name("CUST001"))
print(find_customer_name(None))


Customer-CUST001
None


## 1.3 f-strings

`f"...{expr}..."` embeds Python expressions directly inside a string. This
is how almost every log line and generated response in the project is
built.

In [4]:
name = "Rohan"
amount = 6500.0
print(f"Refund request for {name}, amount \u20b9{amount:.2f} exceeds threshold")
# {amount:.2f} is a *format spec* -- ':.2f' means "2 decimal places"


Refund request for Rohan, amount ₹6500.00 exceeds threshold


Compare to the real line from `escalation_rules.py`:

```python
return True, f"refund request for \u20b9{order_amount:.2f} exceeds the \u20b9{HIGH_VALUE_REFUND_THRESHOLD:.0f} manual-review threshold"
```

Two f-strings' worth of formatting in one return statement — `:.2f` for
cents, `:.0f` for a whole-number threshold.

## 1.4 Control flow — `if`/`elif`/`else`, early returns

`escalation_rules.py`'s `check_hard_escalation` is a chain of early returns:
check one condition, return immediately if it matches, otherwise fall
through to the next check. This pattern avoids deeply nested `if/else`
blocks and reads top-to-bottom as a priority list.

In [5]:
def classify_urgency(issue_type: str, sentiment: str) -> str:
    if issue_type in ("fraud_suspected", "complaint_escalation"):
        return "critical"
    if sentiment == "angry":
        return "high"
    if issue_type == "payment_issue":
        return "high"
    return "medium"

for case in [("fraud_suspected", "neutral"), ("order_status", "angry"), ("order_status", "neutral")]:
    print(case, "->", classify_urgency(*case))


('fraud_suspected', 'neutral') -> critical
('order_status', 'angry') -> high
('order_status', 'neutral') -> medium


Note `classify_urgency(*case)` — the `*` **unpacks** the tuple `case` into two
separate positional arguments. `*args`-style unpacking shows up again when
you get to `agents.py` and `pipeline.py`.

## 1.5 Default and keyword arguments

`CONFIDENCE_THRESHOLD = 0.75` isn't hardcoded inside the function — it's a
module-level constant referenced from the function, and several project
functions use default parameter values so callers only need to pass what's
different from the common case.

In [ ]:
def decide(confidence: float, threshold: float = 0.75) -> str:
    return "escalate" if confidence < threshold else "resolve"

print(decide(0.4))              # uses the default threshold
print(decide(0.4, threshold=0.3))  # overrides it -- explicit keyword argument
